# 08 Business Recommendations and Dashboard Data Preparation

## Loan Default Prediction and Credit Risk Scorecard

This notebook prepares final business outputs for reporting, dashboarding, and executive communication. It consolidates model evaluation, scorecard results, risk segmentation, and business recommendations.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [3]:
scorecard_output = pd.read_csv("../data/outputs/final_scorecard_output.csv")
risk_band_summary = pd.read_csv("../data/outputs/risk_band_summary.csv")
business_scorecard = pd.read_csv("../data/outputs/business_scorecard.csv")
threshold_results = pd.read_csv("../data/outputs/threshold_tuning_results_display.csv")
evaluation_summary = pd.read_csv("../data/outputs/champion_model_evaluation_summary.csv")
decile_analysis = pd.read_csv("../data/outputs/champion_model_decile_analysis.csv")
combined_model_results = pd.read_csv("../data/outputs/combined_model_results.csv")

scorecard_output.shape

(342392, 41)

In [4]:
total_loans = len(scorecard_output)
observed_default_rate = scorecard_output["actual_default"].mean()
total_defaults = scorecard_output["actual_default"].sum()
avg_model_score = scorecard_output["predicted_pd_tree_model"].mean()

critical_risk_loans = scorecard_output[scorecard_output["risk_band"] == "Critical Risk"].shape[0]
critical_risk_default_rate = scorecard_output.loc[
    scorecard_output["risk_band"] == "Critical Risk",
    "actual_default"
].mean()

high_plus_critical = scorecard_output[
    scorecard_output["risk_band"].isin(["High Risk", "Critical Risk"])
]

high_plus_critical_share = len(high_plus_critical) / total_loans
high_plus_critical_default_rate = high_plus_critical["actual_default"].mean()
high_plus_critical_default_capture = high_plus_critical["actual_default"].sum() / total_defaults

executive_kpis = pd.DataFrame({
    "kpi": [
        "Total scored loans",
        "Observed default rate",
        "Average raw model score",
        "Critical risk portfolio share",
        "Critical risk observed default rate",
        "High + Critical risk portfolio share",
        "High + Critical risk observed default rate",
        "High + Critical risk default capture"
    ],
    "value": [
        total_loans,
        observed_default_rate,
        avg_model_score,
        critical_risk_loans / total_loans,
        critical_risk_default_rate,
        high_plus_critical_share,
        high_plus_critical_default_rate,
        high_plus_critical_default_capture
    ]
})

executive_kpis

,kpi,value
0,Total scored loans,342392.000000
1,Observed default rate,0.212350
2,Average raw model score,0.452933
3,Critical risk portfolio share,0.100002
4,Critical risk observed default rate,0.510718
5,High + Critical risk portfolio share,0.250003
6,High + Critical risk observed default rate,0.410788
7,High + Critical risk default capture,0.483626


In [5]:
executive_kpis_display = executive_kpis.copy()

percentage_kpis = [
    "Observed default rate",
    "Average raw model score",
    "Critical risk portfolio share",
    "Critical risk observed default rate",
    "High + Critical risk portfolio share",
    "High + Critical risk observed default rate",
    "High + Critical risk default capture"
]

executive_kpis_display["display_value"] = executive_kpis_display.apply(
    lambda row: f"{row['value']:.2%}" if row["kpi"] in percentage_kpis else f"{int(row['value']):,}",
    axis=1
)

executive_kpis_display

,kpi,value,display_value
0,Total scored loans,342392.000000,"342,392"
1,Observed default rate,0.212350,21.24%
2,Average raw model score,0.452933,45.29%
3,Critical risk portfolio share,0.100002,10.00%
4,Critical risk observed default rate,0.510718,51.07%
5,High + Critical risk portfolio share,0.250003,25.00%
6,High + Critical risk observed default rate,0.410788,41.08%
7,High + Critical risk default capture,0.483626,48.36%


In [6]:
executive_kpis.to_csv("../data/outputs/dashboard_executive_kpis.csv", index=False)
executive_kpis_display.to_csv("../data/outputs/dashboard_executive_kpis_display.csv", index=False)

In [7]:
risk_band_dashboard = risk_band_summary.copy()

risk_band_dashboard = risk_band_dashboard[
    [
        "risk_band",
        "loan_count",
        "default_count",
        "portfolio_share_pct",
        "observed_default_rate_pct",
        "avg_model_score_pct",
        "min_model_score_pct",
        "max_model_score_pct",
        "recommended_action",
        "monitoring_requirement"
    ]
]

risk_band_dashboard

,risk_band,loan_count,default_count,portfolio_share_pct,observed_default_rate_pct,avg_model_score_pct,min_model_score_pct,max_model_score_pct,recommended_action,monitoring_requirement
0,Low Risk,171195,17703,50.0,10.34,28.91,3.18,45.20,Standard approval path; routine monitoring,Monthly portfolio monitoring
1,Medium Risk,85598,19841,25.0,23.18,52.55,45.20,60.29,Standard review; monitor risk indicators,Monthly segment monitoring
2,High Risk,51359,17676,15.0,34.42,65.88,60.29,72.10,Manual underwriting review recommended,Weekly/monthly review of default trends
3,Critical Risk,34240,17487,10.0,51.07,78.16,72.10,92.49,"Senior review, additional documentation, or de...",Close monitoring and policy review


In [8]:
risk_band_dashboard.to_csv("../data/outputs/dashboard_risk_band_summary.csv", index=False)

In [9]:
model_performance_dashboard = combined_model_results.copy()

model_performance_dashboard[
    ["roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1_score"]
] = model_performance_dashboard[
    ["roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1_score"]
].round(4)

model_performance_dashboard

,model,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1_score
0,LightGBM,0.5,0.7287,0.4196,0.6594,0.3460,0.6788,0.4584
1,XGBoost,0.5,0.7275,0.4184,0.7941,0.5768,0.1139,0.1902
2,HistGradientBoosting,0.5,0.7267,0.4172,0.7941,0.5772,0.1128,0.1887
3,Logistic Regression,0.5,0.7161,0.4000,0.7916,0.5597,0.0878,0.1518
4,Regularized Weighted Logistic Regression,0.5,0.7161,0.3992,0.6446,0.3343,0.6799,0.4483
5,Weighted Logistic Regression,0.5,0.7161,0.3992,0.6447,0.3344,0.6797,0.4483
6,Random Forest,0.5,0.7134,0.3966,0.6372,0.3293,0.6834,0.4444
7,Decision Tree,0.5,0.7060,0.3783,0.6646,0.3413,0.6231,0.4410
8,Naive Baseline,0.5,0.5000,0.2124,0.7876,0.0000,0.0000,0.0000


In [11]:
model_performance_dashboard

,model,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1_score
0,LightGBM,0.5,0.7287,0.4196,0.6594,0.3460,0.6788,0.4584
1,XGBoost,0.5,0.7275,0.4184,0.7941,0.5768,0.1139,0.1902
2,HistGradientBoosting,0.5,0.7267,0.4172,0.7941,0.5772,0.1128,0.1887
3,Logistic Regression,0.5,0.7161,0.4000,0.7916,0.5597,0.0878,0.1518
4,Regularized Weighted Logistic Regression,0.5,0.7161,0.3992,0.6446,0.3343,0.6799,0.4483
5,Weighted Logistic Regression,0.5,0.7161,0.3992,0.6447,0.3344,0.6797,0.4483
6,Random Forest,0.5,0.7134,0.3966,0.6372,0.3293,0.6834,0.4444
7,Decision Tree,0.5,0.7060,0.3783,0.6646,0.3413,0.6231,0.4410
8,Naive Baseline,0.5,0.5000,0.2124,0.7876,0.0000,0.0000,0.0000


In [9]:
threshold_dashboard = threshold_results.copy()

threshold_dashboard

,threshold,accuracy,precision,recall,f1_score,true_positives,false_positives,true_negatives,false_negatives,manual_review_count,manual_review_rate,default_capture_rate
0,0.30,43.00,26.15,92.30,40.75,67105,189550,80135,5602,256655,74.96,92.30
1,0.35,49.05,27.85,88.01,42.32,63988,165736,103949,8719,229724,67.09,88.01
2,0.40,54.94,29.76,82.52,43.75,59999,141587,128098,12708,201586,58.88,82.52
3,0.45,60.66,32.02,75.94,45.05,55214,117203,152482,17493,172417,50.36,75.94
4,0.50,65.94,34.60,67.88,45.84,49357,93275,176410,23350,142632,41.66,67.88
5,0.55,70.42,37.46,58.71,45.74,42685,71259,198426,30022,113944,33.28,58.71
6,0.60,74.11,40.86,48.96,44.55,35599,51526,218159,37108,87125,25.45,48.96
7,0.65,76.79,44.64,38.77,41.50,28185,34951,234734,44522,63136,18.44,38.77
8,0.70,78.51,48.96,28.46,35.99,20691,21569,248116,52016,42260,12.34,28.46


In [10]:
threshold_dashboard.to_csv("../data/outputs/dashboard_threshold_analysis.csv", index=False)

In [11]:
decile_dashboard = decile_analysis[
    [
        "risk_decile",
        "loan_count",
        "default_count",
        "avg_predicted_pd_pct",
        "observed_default_rate_pct",
        "default_capture_rate_pct",
        "cumulative_default_capture_rate_pct",
        "lift"
    ]
].copy()

decile_dashboard

,risk_decile,loan_count,default_count,avg_predicted_pd_pct,observed_default_rate_pct,default_capture_rate_pct,cumulative_default_capture_rate_pct,lift
0,1,34240,17487,78.16,51.07,24.05,24.05,2.41
1,2,34239,12452,67.81,36.37,17.13,41.18,1.71
2,3,34239,9912,60.35,28.95,13.63,54.81,1.36
3,4,34239,8303,53.97,24.25,11.42,66.23,1.14
4,5,34239,6850,48.08,20.01,9.42,75.65,0.94
5,6,34239,5574,42.28,16.28,7.67,83.32,0.77
6,7,34239,4657,36.28,13.60,6.41,89.72,0.64
7,8,34239,3569,29.94,10.42,4.91,94.63,0.49
8,9,34239,2571,22.78,7.51,3.54,98.17,0.35
9,10,34240,1332,13.29,3.89,1.83,100.00,0.18


In [12]:
decile_dashboard.to_csv("../data/outputs/dashboard_decile_analysis.csv", index=False)

In [13]:
borrower_profile_summary = (
    scorecard_output.groupby("risk_band")
    .agg(
        loan_count=("actual_default", "count"),
        avg_loan_amount=("loan_amnt", "mean"),
        avg_annual_income=("annual_inc", "mean"),
        avg_dti=("dti", "mean"),
        avg_interest_rate=("int_rate", "mean"),
        avg_revol_util=("revol_util", "mean"),
        avg_credit_history_months=("credit_history_months", "mean"),
        observed_default_rate=("actual_default", "mean")
    )
    .reset_index()
)

borrower_profile_summary["observed_default_rate_pct"] = (
    borrower_profile_summary["observed_default_rate"] * 100
).round(2)

borrower_profile_summary[
    [
        "avg_loan_amount",
        "avg_annual_income",
        "avg_dti",
        "avg_interest_rate",
        "avg_revol_util",
        "avg_credit_history_months"
    ]
] = borrower_profile_summary[
    [
        "avg_loan_amount",
        "avg_annual_income",
        "avg_dti",
        "avg_interest_rate",
        "avg_revol_util",
        "avg_credit_history_months"
    ]
].round(2)

borrower_profile_summary

,risk_band,loan_count,avg_loan_amount,avg_annual_income,avg_dti,avg_interest_rate,avg_revol_util,avg_credit_history_months,observed_default_rate,observed_default_rate_pct
0,Critical Risk,34240,18710.59,63529.94,22.93,20.49,55.95,180.35,0.510718,51.07
1,High Risk,51359,16698.71,67873.07,20.77,16.91,55.61,186.94,0.344166,34.42
2,Low Risk,171195,12982.13,81270.60,15.87,10.22,48.24,202.80,0.103408,10.34
3,Medium Risk,85598,14284.15,69794.53,19.21,14.32,54.53,188.47,0.231793,23.18


In [14]:
borrower_profile_summary.to_csv("../data/outputs/dashboard_borrower_profile_summary.csv", index=False)

In [15]:
purpose_risk_summary = (
    scorecard_output.groupby("purpose")
    .agg(
        loan_count=("actual_default", "count"),
        default_count=("actual_default", "sum"),
        observed_default_rate=("actual_default", "mean"),
        avg_model_score=("predicted_pd_tree_model", "mean")
    )
    .reset_index()
)

purpose_risk_summary["observed_default_rate_pct"] = (
    purpose_risk_summary["observed_default_rate"] * 100
).round(2)

purpose_risk_summary["avg_model_score_pct"] = (
    purpose_risk_summary["avg_model_score"] * 100
).round(2)

purpose_risk_summary = purpose_risk_summary.sort_values(
    "observed_default_rate",
    ascending=False
)

purpose_risk_summary

,purpose,loan_count,default_count,observed_default_rate,avg_model_score,observed_default_rate_pct,avg_model_score_pct
11,small_business,3927,1219,0.310415,0.585885,31.04,58.59
8,moving,2391,568,0.237558,0.491275,23.76,49.13
7,medical,3921,908,0.231574,0.471535,23.16,47.15
9,other,19953,4552,0.228136,0.475572,22.81,47.56
10,renewable_energy,235,53,0.225532,0.485938,22.55,48.59
2,debt_consolidation,198760,44334,0.223053,0.471767,22.31,47.18
5,house,1887,417,0.220986,0.516494,22.10,51.65
12,vacation,2285,480,0.210066,0.430629,21.01,43.06
3,educational,86,17,0.197674,0.390217,19.77,39.02
6,major_purchase,7562,1487,0.196641,0.422567,19.66,42.26


In [16]:
purpose_risk_summary.to_csv("../data/outputs/dashboard_purpose_risk_summary.csv", index=False)

In [17]:
grade_risk_summary = (
    scorecard_output.groupby("grade")
    .agg(
        loan_count=("actual_default", "count"),
        default_count=("actual_default", "sum"),
        observed_default_rate=("actual_default", "mean"),
        avg_model_score=("predicted_pd_tree_model", "mean")
    )
    .reset_index()
)

grade_risk_summary["observed_default_rate_pct"] = (
    grade_risk_summary["observed_default_rate"] * 100
).round(2)

grade_risk_summary["avg_model_score_pct"] = (
    grade_risk_summary["avg_model_score"] * 100
).round(2)

grade_risk_summary = grade_risk_summary.sort_values("grade")

grade_risk_summary

,grade,loan_count,default_count,observed_default_rate,avg_model_score,observed_default_rate_pct,avg_model_score_pct
0,A,59005,3972,0.067316,0.193426,6.73,19.34
1,B,99671,14461,0.145087,0.367320,14.51,36.73
2,C,97614,23177,0.237435,0.519746,23.74,51.97
3,D,51375,16457,0.320331,0.616249,32.03,61.62
4,E,24215,9727,0.401693,0.692913,40.17,69.29
5,F,8099,3669,0.453019,0.744855,45.30,74.49
6,G,2413,1244,0.515541,0.766947,51.55,76.69


In [18]:
grade_risk_summary.to_csv("../data/outputs/dashboard_grade_risk_summary.csv", index=False)

In [19]:
business_recommendations = pd.DataFrame({
    "recommendation_area": [
        "Manual Review",
        "Risk Segmentation",
        "Portfolio Monitoring",
        "Model Calibration",
        "Model Governance",
        "Threshold Management",
        "Fairness and Responsible Use"
    ],
    "recommendation": [
        "Use the High Risk and Critical Risk segments to prioritize manual underwriting review.",
        "Use percentile-based risk bands because raw model probabilities are not fully calibrated.",
        "Monitor default rates by risk band, grade, purpose, term, and origination period.",
        "Calibrate predicted probabilities before using the score as a true probability of default.",
        "Document model assumptions, limitations, intended use, and monitoring requirements before production use.",
        "Select classification thresholds based on review capacity, credit loss tolerance, and business policy.",
        "Avoid using the model for automatic rejection without fairness testing, human review, and governance approval."
    ],
    "business_value": [
        "Improves prioritization of limited underwriting review resources.",
        "Creates clear borrower segments for reporting and decision support.",
        "Helps detect portfolio deterioration and risk concentration.",
        "Improves reliability of PD estimates for business use.",
        "Reduces model risk and improves transparency.",
        "Balances default detection with operational workload.",
        "Reduces risk of unfair or inappropriate credit decisions."
    ]
})

business_recommendations

,recommendation_area,recommendation,business_value
0,Manual Review,Use the High Risk and Critical Risk segments t...,Improves prioritization of limited underwritin...
1,Risk Segmentation,Use percentile-based risk bands because raw mo...,Creates clear borrower segments for reporting ...
2,Portfolio Monitoring,"Monitor default rates by risk band, grade, pur...",Helps detect portfolio deterioration and risk ...
3,Model Calibration,Calibrate predicted probabilities before using...,Improves reliability of PD estimates for busin...
4,Model Governance,"Document model assumptions, limitations, inten...",Reduces model risk and improves transparency.
5,Threshold Management,Select classification thresholds based on revi...,Balances default detection with operational wo...
6,Fairness and Responsible Use,Avoid using the model for automatic rejection ...,Reduces risk of unfair or inappropriate credit...


In [20]:
business_recommendations.to_csv("../data/outputs/business_recommendations_table.csv", index=False)

# Business Recommendations

## 1. Use the Model for Manual Review Prioritization

The model should be used as a decision-support tool to prioritize borrower applications for manual review. It should not be used as a fully automated approval or decline engine.

The High Risk and Critical Risk segments should receive additional underwriting attention because they show materially higher observed default rates.

## 2. Use Risk Bands for Portfolio Monitoring

The percentile-based risk bands provide a practical way to monitor portfolio risk. The business should track loan volume, default rate, exposure, and borrower characteristics by risk band over time.

## 3. Monitor High and Critical Risk Segments Closely

The High Risk and Critical Risk segments represent a smaller share of the scored population but contain a disproportionate share of observed defaults. These segments should be monitored for changes in default rate, approval volume, loan purpose, and borrower profile.

## 4. Calibrate Probabilities Before Production Use

The champion model performs well as a ranking model, but calibration analysis showed that raw predicted scores overestimate observed default rates. Before using the output as a true probability of default, the model should be calibrated and validated on current internal lending data.

## 5. Select Thresholds Based on Business Capacity

The 0.50 threshold captures a meaningful share of defaults but also sends a large share of borrowers to manual review. In production, the threshold should be selected based on underwriting capacity, credit loss tolerance, expected approval impact, and compliance requirements.

## 6. Maintain Model Governance

Before production use, the model should have a documented model card, validation results, monitoring plan, retraining schedule, and responsible-use guidelines.

## 7. Avoid Automatic Decline Decisions

The model should not be used to automatically decline borrowers without human review, fairness testing, policy validation, and governance approval.

# Business Recommendations Summary

## Key Actions Completed

- Created executive KPI outputs.
- Prepared dashboard-ready model performance tables.
- Prepared risk band and scorecard tables.
- Prepared borrower profile summaries by risk band.
- Prepared loan purpose and grade risk summaries.
- Created business recommendations for manual review, monitoring, calibration, threshold management, and governance.
- Saved dashboard-ready CSV files for Power BI/Tableau.

## Next Step

The next phase will create SQL scripts and professional documentation, including the model card, executive summary, and dashboard design.